In [ ]:
import gymnasium as gym
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import numpy as np
import matplotlib.pyplot as plt

envs = gym.make('FrozenLake-v1', desc=generate_random_map(size=8))

### 贝尔曼方程
$$
V(s) = \sum_{a} \pi(a|s) \sum_{r,s^{\prime}} p(r,s^{\prime}|s,a)(r+\gamma V(s^{\prime}))
$$
$$
Q(s,a) = \sum_{r,s^{\prime}} p(r,s^{\prime}|s,a)(r+\gamma V(s^{\prime}))
$$
### 迭代公式
$$
V_{k+1}(s) = \sum_{a} \pi(a|s) \sum_{r,s^{\prime}} p(r,s^{\prime}|s,a)(r+\gamma V_k(s^{\prime}))
$$
### 贝尔曼最优公式
$$
V_{k+1}(s) = \max_{a}\sum_{r,s^{\prime}} p(r,s^{\prime}|s,a)(r+\gamma V_k(s^{\prime}))
$$

In [ ]:
# 策略迭代
def policy_evaluation(policy , p , gamma = 1 , theta = 1e-10):
    pre_v = np.zeros(len(p) , dtype=np.float64)

    while True:

        v = np.zeros(len(pre_v) , dtype=np.float64)

        for state in range(len(v)):

            for prob , next_state , reward , done in p[state][policy[state]]:

                v[state] += prob * (reward + gamma * pre_v[next_state] * (not done))

        if np.max(np.abs(v - pre_v)) < theta:
            break

        pre_v = v.copy()

    return v


def policy_improvement(v , p , gamma = 1):
    q = np.zeros((len(p), len(p[0])) , dtype=np.float64)

    for state in range(len(p)):
        for action in range(len(p[0])):
            for prob , next_state , reward , done in p[state][action]:
                q[state][action] += prob * (reward + gamma * v[next_state] * (not done))


    return np.argmax(q , axis=1)

def policy_iteration(p , gamma = 1 , theta = 1e-10 , max_episode = 100000):

    init_action = np.random.randint(low=0,high=len(p[0]),size=len(p))

    i = 0

    while True:
        v = policy_evaluation(init_action , p , gamma, theta)
        policy = policy_improvement(v , p , gamma)

        i += 1

        if i > max_episode or np.array_equal(policy, init_action):
            break
        init_action = policy.copy()
    return init_action , i


In [ ]:
# 价值迭代
def value_iteration(p , gamma = 1 , theta = 1e-10 , max_episode = 100000):
    v = np.zeros(len(p) , dtype=np.float64)

    i = 0

    while True:

        q = np.zeros((len(p), len(p[0])) , dtype=np.float64)

        for state in range(len(p)):
            for action in range(len(p[0])):
                for prob , next_state , reward , done in p[state][action]:
                    q[state][action] += prob * (reward + gamma * v[next_state] * (not done))
        i += 1
        v_new = np.max(q , axis=1)

        if i > max_episode or np.max(np.abs(v - v_new)) < theta:
            v = v_new.copy()
            break

        v = v_new.copy()

    pi = np.argmax(q , axis=1)

    return pi , i


In [ ]:
import time

def run_dp(env, algorithm, gamma=1, theta=1e-6 , max_episode = 10000):
    env.reset()
    p = env.unwrapped.P

    start = time.time()

    policy , i = algorithm(
        p,
        gamma,
        theta,
        max_episode
    )

    end = time.time()

    print(f'iteration num: {i} , 运行时间: {end - start} s')

    return policy

def visualize_policy(policy, env):
    desc = env.unwrapped.desc

    nrow, ncol = desc.shape

    action_map = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    fig, ax = plt.subplots(figsize=(5, 5))

    for state, action in enumerate(policy):

        row = state // ncol
        col = state % ncol

        cell = desc[row][col].decode("utf-8")

        # 不显示洞和终点动作
        if cell in ["H", "G"]:
            continue

        ax.text(
            col,
            row,
            action_map[action],
            ha="center",
            va="center",
            fontsize=25
        )

    for i in range(nrow + 1):
        ax.axhline(i - 0.5)

    for j in range(ncol + 1):
        ax.axvline(j - 0.5)

    ax.set_xticks([])
    ax.set_yticks([])

    plt.show()

In [ ]:
policy = run_dp(envs , policy_iteration , gamma = 0.98)
visualize_policy(policy, envs)

policy = run_dp(envs , value_iteration , gamma = 0.98)
visualize_policy(policy, envs)